<a href="https://colab.research.google.com/github/ataulhaque/ML/blob/main/mini_gpt_torch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# -----------------------
# Load data
# -----------------------

with open("/content/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# -----------------------
# Hyperparameters
# -----------------------

n_embd = 128
n_head = 4
n_layer = 2
block_size = 128
batch_size = 32
num_steps = 8000
learning_rate = 3e-4

device = "cpu"

# -----------------------
# Model
# -----------------------

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=n_embd,
                nhead=n_head,
                dim_feedforward=4*n_embd,
                activation="relu",
                batch_first=True
            )
            for _ in range(n_layer)
        ])

        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        B, T = x.shape

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))

        x = tok + pos

        # 🔥 Create causal mask (T x T)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))

        for block in self.blocks:
            x = block(x, src_mask=mask)

        x = self.ln(x)
        logits = self.head(x)

        return logits


model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# -----------------------
# Training
# -----------------------

for step in range(num_steps):

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)

    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

# -----------------------
# Generation
# -----------------------

print("\n--- Generated Text ---")

context = torch.zeros((1, 1), dtype=torch.long).to(device)

temperature = 0.8
top_k = 40

for _ in range(300):

    logits = model(context[:, -block_size:])
    logits = logits[:, -1, :] / temperature

    # 🔥 top-k filtering
    values, indices = torch.topk(logits, top_k)
    probs = F.softmax(values, dim=-1)

    next_token = indices.gather(-1, torch.multinomial(probs, 1))

    context = torch.cat((context, next_token), dim=1)

print("".join([itos[i.item()] for i in context[0]]))


step 0 | loss 4.9263
step 100 | loss 2.9035
step 200 | loss 2.6917
step 300 | loss 2.5960
step 400 | loss 2.5485
step 500 | loss 2.5301
step 600 | loss 2.5452
step 700 | loss 2.4597
step 800 | loss 2.4226
step 900 | loss 2.4385
step 1000 | loss 2.3760
step 1100 | loss 2.3160
step 1200 | loss 2.2302
step 1300 | loss 2.2505
step 1400 | loss 2.2312
step 1500 | loss 2.1588
step 1600 | loss 2.1915
step 1700 | loss 2.1363
step 1800 | loss 2.1575
step 1900 | loss 2.1557
step 2000 | loss 2.1234
step 2100 | loss 2.1153
step 2200 | loss 2.0559
step 2300 | loss 2.0456
step 2400 | loss 2.0301
step 2500 | loss 2.0050
step 2600 | loss 1.9860
step 2700 | loss 2.0388
step 2800 | loss 1.9806
step 2900 | loss 2.0360
step 3000 | loss 1.9846
step 3100 | loss 1.9746
step 3200 | loss 1.9455
step 3300 | loss 1.9987
step 3400 | loss 1.9623
step 3500 | loss 1.9808
step 3600 | loss 1.9494
step 3700 | loss 1.9132
step 3800 | loss 1.9917
step 3900 | loss 1.8877
step 4000 | loss 1.9058
step 4100 | loss 1.8994
step